# اليوم 3، المعمل 4: التقييم والعتبة

لأن 14% فقط من العملاء يتوقفون، نستخدم PR-AUC للمقارنة. ثم نختار عتبة تسمح بالتواصل مع 20% من العملاء فقط.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from manafeth.data import load_customers, split_customers
from manafeth.features import build_preprocessor
from manafeth.evaluation import evaluate_classifier, recall_at_fraction, threshold_for_fraction

df = load_customers(DATA)
X_train, X_test, y_train, y_test = split_customers(df)
model = Pipeline([("prep", build_preprocessor()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])


## Cross-Validation موحد

قيّم Dummy والنموذج بنفس الطيات. اذكر المتوسط والانحراف المعياري.

In [ ]:
dummy = Pipeline([("prep", build_preprocessor()), ("model", DummyClassifier(strategy="prior"))])
comparison = pd.DataFrame({"dummy": evaluate_classifier(dummy, X_train, y_train), "logistic": evaluate_classifier(model, X_train, y_train)}).T
display(comparison)

X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=.25, stratify=y_train, random_state=RANDOM_STATE)
model.fit(X_fit, y_fit)
prob = model.predict_proba(X_valid)[:, 1]
threshold = threshold_for_fraction(prob, .20)
print("threshold:", round(threshold, 3), "recall@20%:", round(recall_at_fraction(y_valid, prob, .20), 3))

rows = []
for city, idx in X_valid.groupby("city").groups.items():
    pred = prob[X_valid.index.get_indexer(idx)] >= threshold
    truth = y_valid.loc[idx].to_numpy()
    rows.append({"city": city, "recall": truth[pred].sum() / max(1, truth.sum()), "churners": int(truth.sum())})
display(pd.DataFrame(rows).set_index("city"))


## عتبة الميزانية

أنشئ validation داخل train. درّب النموذج، استخرج الاحتمالات، ثم احسب threshold وrecall عندما نختار أعلى 20%.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


## تحليل الشرائح

قارن recall بين المدن وبين العملاء الجدد والقدامى. لا يعني الفرق وحده وجود ظلم، لكنه إشارة تحتاج تفسيرًا وبيانات إضافية.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


**ناتج التسليم:** جدول CV، جملة قرار مثل «عند التواصل مع 20% نسترجع ...% من المتسربين»، وجدول شرائح.